In [3]:
import numpy as np
import jax.numpy as jnp
import jax
import optax 
import matplotlib.pyplot as plt
import timeout_decorator
import pickle
from typing import Any, Callable, Dict, Optional, Sequence
from scipy.ndimage import gaussian_filter1d

import sys, os, glob
from typing import Tuple, List

sys.path.insert(0, os.path.abspath('/home/dabin/code/EDGAR'))

# import local modules
import utils, diagnostic, loss_functions, seed_programs, hypothesis_engine

gpu
[CudaDevice(id=0)]


In [ ]:
def three_rep_cvpca(R1, R2, R3, return_evecs=False):
    # R1: n_cells x n_stim
    # R2: n_cells x n_stim
    # R3: n_cells x n_stim
    # return evals: n_stim
    n_cells, n_angles = R1.shape
    R1z = (R1 - R1.mean(axis = 0, keepdims=True)) / (R1.std(axis = 0, keepdims=True, ddof=0) + 1e-10)
    R2z = (R2 - R2.mean(axis = 0, keepdims=True)) / (R2.std(axis = 0, keepdims=True, ddof=0) + 1e-10)
    R3z = (R3 - R3.mean(axis = 0, keepdims=True)) / (R3.std(axis = 0, keepdims=True, ddof=0) + 1e-10)
    C11 = R1z.T @ R1z / n_cells # n_stim x n_stim
    evals, evecs = np.linalg.eigh(C11)
    idx = np.argsort(evals)[::-1]
    evals = evals[idx]
    evecs = evecs[:, idx]
    # project onto C23 
    C23 = R2z.T @ R3z / n_cells # n_stim x n_stim
    evals = np.diag(evecs.T @ C23 @ evecs)
    if return_evecs:
        return evals, evecs
    return evals

In [8]:
neuron_model_2_jax = seed_programs.neuron_model_2_jax
parameter_estimator_2 = seed_programs.parameter_estimator_2

def neuron_model(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_ccw_1=1.0,
                width_cw_1=1.0,
                exponent_1=2.0,
                theta_pref_2=jnp.pi,
                amplitude_2=0.0,
                width_ccw_2=1.0,
                width_cw_2=1.0,
                exponent_2=2.0, ):

    # ---- Positivity/box constraints ----
    width_ccw_1 = jnp.clip(width_ccw_1, 1e-6, None)
    width_cw_1  = jnp.clip(width_cw_1,  1e-6, None)
    width_ccw_2 = jnp.clip(width_ccw_2, 1e-6, None)
    width_cw_2  = jnp.clip(width_cw_2,  1e-6, None)
    exponent_1  = jnp.clip(exponent_1,  0.1,  5.0)
    exponent_2  = jnp.clip(exponent_2,  0.1,  5.0)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    width_1_effective = jnp.where(signed_diff_1 < 0, width_ccw_1, width_cw_1)
    width_1_effective = jnp.maximum(width_1_effective, 1e-6)
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1_effective) ** exponent_1)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    width_2_effective = jnp.where(signed_diff_2 < 0, width_ccw_2, width_cw_2)
    width_2_effective = jnp.maximum(width_2_effective, 1e-6)
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2_effective) ** exponent_2)

    return baseline + peak1_component + peak2_component

In [5]:
def neuron_model_antipodal(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_ccw_1=1.0,
                width_cw_1=1.0,
                exponent_1=2.0,
                amplitude_2=0.0,
                width_ccw_2=1.0,
                width_cw_2=1.0,
                exponent_2=2.0, 
                ):

    theta_pref_2 = jnp.mod(theta_pref_1 + jnp.pi, 2 * jnp.pi)

    # ---- Positivity/box constraints ----
    width_ccw_1 = jnp.clip(width_ccw_1, 1e-6, None)
    width_cw_1  = jnp.clip(width_cw_1,  1e-6, None)
    width_ccw_2 = jnp.clip(width_ccw_2, 1e-6, None)
    width_cw_2  = jnp.clip(width_cw_2,  1e-6, None)
    exponent_1  = jnp.clip(exponent_1,  0.1,  5.0)
    exponent_2  = jnp.clip(exponent_2,  0.1,  5.0)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    width_1_effective = jnp.where(signed_diff_1 < 0, width_ccw_1, width_cw_1)
    width_1_effective = jnp.maximum(width_1_effective, 1e-6)
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1_effective) ** exponent_1)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    width_2_effective = jnp.where(signed_diff_2 < 0, width_ccw_2, width_cw_2)
    width_2_effective = jnp.maximum(width_2_effective, 1e-6)
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2_effective) ** exponent_2)

    return baseline + peak1_component + peak2_component

def neuron_model_antipodal_symmetric(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_1=1.0,
                exponent_1=2.0,
                amplitude_2=0.0,
                width_2=1.0,
                exponent_2=2.0, 
                ):

    theta_pref_2 = jnp.mod(theta_pref_1 + jnp.pi, 2 * jnp.pi)

    # ---- Positivity/box constraints ----
    width_1 = jnp.clip(width_1, 1e-6, None)
    width_2 = jnp.clip(width_2, 1e-6, None)
    exponent_1  = jnp.clip(exponent_1,  0.1,  5.0)
    exponent_2  = jnp.clip(exponent_2,  0.1,  5.0)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1) ** exponent_1)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2) ** exponent_2)

    return baseline + peak1_component + peak2_component

def neuron_model_no_p(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_ccw_1=1.0,
                width_cw_1=1.0,
                theta_pref_2=jnp.pi,
                amplitude_2=0.0,
                width_ccw_2=1.0,
                width_cw_2=1.0,
                ):

    # ---- Positivity/box constraints ----
    width_ccw_1 = jnp.clip(width_ccw_1, 1e-6, None)
    width_cw_1  = jnp.clip(width_cw_1,  1e-6, None)
    width_ccw_2 = jnp.clip(width_ccw_2, 1e-6, None)
    width_cw_2  = jnp.clip(width_cw_2,  1e-6, None)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    width_1_effective = jnp.where(signed_diff_1 < 0, width_ccw_1, width_cw_1)
    width_1_effective = jnp.maximum(width_1_effective, 1e-6)
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1_effective) ** 2.0)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    width_2_effective = jnp.where(signed_diff_2 < 0, width_ccw_2, width_cw_2)
    width_2_effective = jnp.maximum(width_2_effective, 1e-6)
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2_effective) ** 2.0)

    return baseline + peak1_component + peak2_component

def neuron_model_no_p_antipodal(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_ccw_1=1.0,
                width_cw_1=1.0,
                amplitude_2=0.0,
                width_ccw_2=1.0,
                width_cw_2=1.0,
                ):

    # ---- Positivity/box constraints ----
    width_ccw_1 = jnp.clip(width_ccw_1, 1e-6, None)
    width_cw_1  = jnp.clip(width_cw_1,  1e-6, None)
    width_ccw_2 = jnp.clip(width_ccw_2, 1e-6, None)
    width_cw_2  = jnp.clip(width_cw_2,  1e-6, None)
    theta_pref_2 = jnp.mod(theta_pref_1 + jnp.pi, 2 * jnp.pi)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    width_1_effective = jnp.where(signed_diff_1 < 0, width_ccw_1, width_cw_1)
    width_1_effective = jnp.maximum(width_1_effective, 1e-6)
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1_effective) ** 2.0)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    width_2_effective = jnp.where(signed_diff_2 < 0, width_ccw_2, width_cw_2)
    width_2_effective = jnp.maximum(width_2_effective, 1e-6)
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2_effective) ** 2.0)

    return baseline + peak1_component + peak2_component

def neuron_model_no_p_antipodal_individually_symmetric(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_1=1.0,
                amplitude_2=0.0,
                width_2=1.0,
                ):

    # ---- Positivity/box constraints ----
    width_1 = jnp.clip(width_1, 1e-6, None)
    width_2 = jnp.clip(width_2, 1e-6, None)
    theta_pref_2 = jnp.mod(theta_pref_1 + jnp.pi, 2 * jnp.pi)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    eps = 1e-12
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1) ** 2.0)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2) ** 2.0)

    return baseline + peak1_component + peak2_component

In [6]:
ablation_case_neuron_model_dict = {
    0 : neuron_model,
    1 : neuron_model_antipodal,
    2 : neuron_model_antipodal_symmetric,
    3 : neuron_model_no_p,
    4 : neuron_model_no_p_antipodal,
    5 : neuron_model_no_p_antipodal_individually_symmetric,
}

In [10]:
def parameter_estimator_optimised(stimuli, spike_counts, fix_second_peak=False, fix_width_symmetry_per_peak=False, fix_exponent=False,):
    """
    Estimates parameters for the neuron_model_v3 based on stimulus angles and observed spike counts.
    This estimator employs statistical principles to identify and characterize tuning curve peaks,
    including baseline, amplitude, asymmetric widths, preferred directions, and exponents for
    two potential peaks.

    Parameters are estimated directly from a smoothed firing rate curve, aiming for a simple
    yet robust initial estimation suitable for generalized Gaussian-like tuning profiles.

    Parameters:
    stimuli (np.ndarray): An array of stimulus angles in radians (0 to 2*pi).
    spike_counts (np.ndarray): An array of spike counts corresponding to each stimulus.

    Returns:
    np.ndarray: An array containing the estimated parameters in the following order:
                [theta_pref_1, baseline, amplitude_1, width_ccw_1, width_cw_1,
                 exponent_1, theta_pref_2, amplitude_2, width_ccw_2, width_cw_2, exponent_2]
    """
    # --- Configuration Constants ---
    n_bins = 180  # Number of angular bins for tuning curve estimation
    kernel_sigma = 2.5 # Sigma for Gaussian smoothing kernel
    min_peak_amplitude = 0.12 # Min amplitude (spikes/stimulus) above baseline for a peak to be considered valid
    min_model_width = 0.1 # Minimum allowed width for numerical stability (as per model)
    default_width_value = 1.0 # Default width for non-significant peaks or non-determinable widths
    min_exponent = 0.1 # Minimum allowed exponent value
    max_exponent = 5.0 # Maximum allowed exponent value
    default_exponent_value = 2.0 # Default exponent (Gaussian)
    min_second_peak_ratio = 0.1 # Min amplitude of secondary peak relative to primary
    min_second_peak_separation = np.pi / 4 # Min angular separation between primary and secondary peaks

    # --- 1. Binning and Smoothing ---
    # Convert stimuli to bin indices, handle wrap-around implicitly by modulo in bincount
    bin_idx = ((stimuli * n_bins) / (2 * np.pi)).astype(np.int32)
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    sums = np.bincount(bin_idx, weights=spike_counts, minlength=n_bins)
    counts = np.bincount(bin_idx, minlength=n_bins)

    # Create Gaussian smoothing kernel
    kernel_radius = int(3 * kernel_sigma)
    x_kernel = np.arange(-kernel_radius, kernel_radius + 1)
    kernel = np.exp(-0.5 * (x_kernel / kernel_sigma) ** 2)
    kernel /= (np.sum(kernel) + 1e-8) # Normalize kernel

    # Pad arrays for circular convolution
    pad = len(kernel) // 2
    sums_padded = np.pad(sums, (pad, pad), mode='wrap')
    counts_padded = np.pad(counts, (pad, pad), mode='wrap')

    # Convolve to get smoothed sum of spikes and counts
    num_conv = np.convolve(sums_padded, kernel, mode='valid')
    den_conv = np.convolve(counts_padded, kernel, mode='valid')

    # Calculate smoothed tuning curve (avoid division by zero)
    tuning_curve = np.zeros_like(num_conv, dtype=float)
    valid_den_mask = den_conv > 1e-8
    tuning_curve[valid_den_mask] = num_conv[valid_den_mask] / den_conv[valid_den_mask]

    angle_step = 2 * np.pi / n_bins

    # --- 2. Baseline Estimation ---
    # Baseline cannot be negative
    baseline_est = np.maximum(0.0, np.min(tuning_curve))
    bin_centers = np.linspace(angle_step/2, 2*np.pi - angle_step/2, n_bins)

    # --- Helper function for estimating peak shape parameters ---
    def _get_peak_params_simple(peak_idx_val, peak_idx, bsl, tc, n_bns, ang_step,
                                min_w, def_w, min_exp, max_exp, def_exp, min_amp_thresh, width_ccw_override=None, width_cw_override=None, theta_pref=None):
        amp = peak_idx_val - bsl
        if amp < min_amp_thresh:
            return amp, def_w, def_w, def_exp # Return default parameters if amplitude too small

        if width_ccw_override is None and width_cw_override is None:
            # Find bins where tuning curve drops below half-max
            target_half_val = bsl + amp / 2.0
            half_ccw_bins, half_cw_bins = 0, 0
            for k in range(1, n_bns // 2 + 1):
                if half_ccw_bins == 0 and tc[(peak_idx - k + n_bns) % n_bns] <= target_half_val:
                    half_ccw_bins = k
                if half_cw_bins == 0 and tc[(peak_idx + k) % n_bns] <= target_half_val:
                    half_cw_bins = k
                if half_ccw_bins > 0 and half_cw_bins > 0:
                    break

            if fix_exponent:
                exponent = default_exponent_value
                SQRT_2_LOG_2 = np.sqrt(2 * np.log(2)) 
                width_ccw = (half_ccw_bins * ang_step) / SQRT_2_LOG_2 if half_ccw_bins > 0 else def_w
                width_cw = (half_cw_bins * ang_step) / SQRT_2_LOG_2 if half_cw_bins > 0 else def_w

                if fix_width_symmetry_per_peak:
                    width_avg = np.mean([width_ccw, width_cw])            
                    width_ccw = width_cw = width_avg
                
                return amp, width_ccw, width_cw, exponent

            else:
                # OPTIMISED
                p_grid = np.linspace(1.0, 2.0, 10)
                P_POWER_2_LOG_2 = (2 * np.log(2)) ** (1 / p_grid)
                width_ccw = np.where(half_ccw_bins > 0,
                    (half_ccw_bins * ang_step) / P_POWER_2_LOG_2,
                    def_w
                )
                width_cw = np.where(half_cw_bins > 0,
                    (half_cw_bins * ang_step) / P_POWER_2_LOG_2,
                    def_w
                )

                width_ccw = np.clip(width_ccw, min_w, np.pi)
                width_cw  = np.clip(width_cw, min_w, np.pi)  

                # Exponent estimation (log(2) / log(dist_qtr / dist_half))
                preds = neuron_model(bin_centers[None, :], theta_pref, bsl, amp, width_ccw[:, None], width_cw[:, None], p_grid[:, None])
                sses = np.sum((tuning_curve - preds) ** 2, axis=1)

                best_idx = np.argmin(sses)
                best = (sses[best_idx], float(p_grid[best_idx]), float(width_ccw[best_idx]), float(width_cw[best_idx]))

                _, p_star, width_ccw, width_cw = best
                exponent = np.clip(p_star, min_exp, max_exp)

                if fix_width_symmetry_per_peak:
                    width_avg = np.mean([width_ccw, width_cw])            
                    width_ccw = width_cw = width_avg

        else:
            # If width overrides are provided, use them directly
            width_ccw = width_ccw_override
            width_cw = width_cw_override
        
        return amp, width_ccw, width_cw, exponent

    # --- 3. Peak Identification and Parameter Estimation ---
    # Find all local maxima in the smoothed tuning curve
    local_maxima = []
    for i in range(n_bins):
        prev_val = tuning_curve[(i - 1 + n_bins) % n_bins]
        next_val = tuning_curve[(i + 1) % n_bins]
        if tuning_curve[i] >= prev_val and tuning_curve[i] >= next_val:
            local_maxima.append((tuning_curve[i], i))
    local_maxima.sort(key=lambda x: x[0], reverse=True) # Sort by peak amplitude (descending)

    # Initialize all output parameters with model defaults (if no peak is found)
    theta_pref_1 = 0.0
    amplitude_1 = default_width_value # amplitude for primary peak default to 1 for non-zero contribution if peak is small
    width_ccw_1, width_cw_1, exponent_1 = default_width_value, default_width_value, default_exponent_value
    
    theta_pref_2 = np.pi # Antipodal to default theta_pref_1
    amplitude_2 = 0.0 # Default no second peak
    width_ccw_2, width_cw_2, exponent_2 = default_width_value, default_width_value, default_exponent_value

    # Process Primary Peak
    if local_maxima:
        peak_1_val, peak_1_idx = local_maxima[0]
        # Estimate parameters for the primary (highest) peak
        theta_pref_1 = peak_1_idx * angle_step
        amplitude_1, width_ccw_1, width_cw_1, exponent_1 = \
            _get_peak_params_simple(peak_1_val, peak_1_idx, baseline_est, tuning_curve, n_bins, angle_step,
                                    min_model_width, default_width_value, min_exponent, max_exponent,
                                    default_exponent_value, min_peak_amplitude, theta_pref=theta_pref_1)
        
        if len(local_maxima) > 1:
            if fix_second_peak:
                # If fix_second_peak is True, we only estimate the primary peak and let theta_pref_2 be antipodal
                # estimate amplitude, width_ccw_, width_cw, exponent_2
                theta_pref_2 = (theta_pref_1 + np.pi) % (2 * np.pi)
                peak_2_idx = int(theta_pref_2 / angle_step) % n_bins
                peak_2_val = tuning_curve[peak_2_idx]

                amplitude_2, width_ccw_2, width_cw_2, exponent_2 = \
                    _get_peak_params_simple(peak_2_val, peak_2_idx, baseline_est, tuning_curve, n_bins, angle_step,
                                            min_model_width, default_width_value, min_exponent, max_exponent,
                                            default_exponent_value, min_peak_amplitude, theta_pref=theta_pref_2)
    
            else:
                # Iterate through other local maxima to find a suitable secondary peak
                for i in range(1, len(local_maxima)):
                    peak_2_val_candidate, peak_2_idx_candidate = local_maxima[i]
                    current_amplitude_2_candidate = peak_2_val_candidate - baseline_est
                    current_theta_pref_2_candidate = peak_2_idx_candidate * angle_step
                    
                    # Check amplitude significance relative to primary peak's estimated amplitude
                    if current_amplitude_2_candidate < (amplitude_1 * min_second_peak_ratio):
                        continue # Skip if too small compared to the primary peak
                    
                    # Check angular separation from the *primary* peak
                    # np.arctan2(sin(a-b), cos(a-b)) gives circular distance in [-pi, pi]
                    peak_sep_rad = np.abs(np.arctan2(np.sin(theta_pref_1 - current_theta_pref_2_candidate),
                                                    np.cos(theta_pref_1 - current_theta_pref_2_candidate)))
                    if peak_sep_rad < min_second_peak_separation:
                        continue # Skip if too close to the primary peak

                    # Valid secondary peak found, estimate its parameters
                    amplitude_2, width_ccw_2, width_cw_2, exponent_2 = \
                        _get_peak_params_simple(peak_2_val_candidate, peak_2_idx_candidate, baseline_est, tuning_curve, n_bins, angle_step,
                                                min_model_width, default_width_value, min_exponent, max_exponent,
                                                default_exponent_value, min_peak_amplitude, theta_pref=current_theta_pref_2_candidate)
                    theta_pref_2 = current_theta_pref_2_candidate
                    
                    # Break after finding the first suitable secondary peak (highest amplitude among remaining candidates)
                    break
        
            # check that ablations were implemented correctly 
            if fix_width_symmetry_per_peak and width_cw_2 is not None and width_ccw_2 is not None:
                assert width_ccw_1 == width_cw_1, "Width symmetry is supposed to be preserved but the first widths are not equal."
                assert width_ccw_2 == width_cw_2, "Width symmetry is supposed to be preserved but the second widths are not equal."

    # --- Pack and return parameters in the correct order for neuron_model_v3 ---
    return np.array([theta_pref_1, baseline_est, amplitude_1, width_ccw_1, width_cw_1,
                 exponent_1, theta_pref_2, amplitude_2, width_ccw_2, width_cw_2, exponent_2])
    

In [11]:
def make_explained_var_score_on_means(model):
    def _score(params, angles, response):  # responses: (n_cells, n_stims)
        # angles: (n_stims,)
        preds = jax.vmap(lambda p: model(angles, *p))(params)  # (n_cells, n_stims)
        ss_res = jnp.sum((response - preds) ** 2, axis=1)
        ss_tot = jnp.sum((response - response.mean(axis=1, keepdims=True)) ** 2, axis=1)
        return 1.0 - ss_res / jnp.maximum(ss_tot, 1e-8)
    return jax.jit(_score)

explained_var_ai = make_explained_var_score_on_means(neuron_model)
explained_var_gaussian = make_explained_var_score_on_means(neuron_model_2_jax)

In [38]:
def train_generalised(
    x_data: jnp.ndarray,
    y_data: jnp.ndarray,
    neuron_model: Callable,
    parameter_estimator: Callable,
    loss_function: Callable,
    val1_fraction: float = 0,
    val2_fraction: float = 0,
    num_steps: int = 6_000,
    learning_rate: float = 1e-3,
    print_every: int = 100,
    manual_exponents: jnp.ndarray = None,
    seed: int = 42,
    cellwise_patience: bool = True,  # NEW
    debug: bool = False,
    ablation_case: int = 0,
    default_init_params: jnp.ndarray = None,
    return_params_init=False,
) -> Dict[str, Any]:
    """
    Train with two validation sets

    - val1_fraction: used for early-stopping step (per cell)
    - val2_fraction: used for patience tuning (per cell or global)
    - cellwise_patience: if True, pick patience per cell; else one global patience
    """
    rng = np.random.default_rng(seed)
    n_trials = x_data.shape[1]
    
    if debug : 
        print(f'x_data shape : {x_data.shape}, y_data shape : {y_data.shape}')

    # ---------------------------
    # Split indices
    # ---------------------------
    if val1_fraction < 0 or val2_fraction < 0 or (val1_fraction + val2_fraction) >= 1:
        raise ValueError("Require 0 ≤ val1_frac, val2_frac and val1_frac + val2_frac < 1.")

    if val1_fraction == 0 and val2_fraction == 0:
        training_trials = np.arange(n_trials)
        val1_trials = np.array([], dtype=int)
        val2_trials = np.array([], dtype=int)
    else:
        n_val1 = int(np.floor(n_trials * val1_fraction))
        n_val2 = int(np.floor(n_trials * val2_fraction))
        all_idx = np.arange(n_trials)
        rng.shuffle(all_idx)
        val2_trials = all_idx[:n_val2]
        val1_trials = all_idx[n_val2:n_val2 + n_val1]
        training_trials = all_idx[n_val2 + n_val1:]

    print(f"Data split: {len(training_trials)} train, {len(val1_trials)} val1, {len(val2_trials)} val2")

    x_train, y_train = x_data[:, training_trials], y_data[:, training_trials]
    x_val1 = x_data[:, val1_trials] if val1_fraction > 0 else None
    y_val1 = y_data[:, val1_trials] if val1_fraction > 0 else None
    x_val2 = x_data[:, val2_trials] if val2_fraction > 0 else None
    y_val2 = y_data[:, val2_trials] if val2_fraction > 0 else None

    # ---------------------------
    # Loss helpers
    # ---------------------------
    loss_single = lambda p, x, y: jnp.nanmean(loss_function(neuron_model(x, *p), y), axis=-1)
    loss_total = jax.vmap(loss_single, in_axes=(0, 0, 0), out_axes=0)

    @jax.jit
    def mean_loss(params, x, y):
        return jnp.nanmean(loss_total(params, x, y))

    # ---------------------------
    # Optimizer
    # ---------------------------
    def make_train_fns(x, y):
        obj = lambda p: mean_loss(p, x, y)
        obj_and_grad = jax.value_and_grad(obj)
        opt = optax.adam(learning_rate)
        def init_state(p0): return opt.init(p0)
        @jax.jit
        def step(p, opt_state):
            loss_val, grads = obj_and_grad(p)
            updates, opt_state = opt.update(grads, opt_state, p)
            p = optax.apply_updates(p, updates)
            return p, opt_state, loss_val
        return init_state, step

    # ---------------------------
    # Init params
    # ---------------------------
    if ablation_case == 0:
        fix_exponent = fix_second_peak = fix_width_symmetry_per_peak = False
    elif ablation_case == 1:
        # print("Ablation case 1: second peak orientation set to first peak + pi (mod 2pi).")
        fix_exponent = fix_width_symmetry_per_peak = False
        fix_second_peak = True
    elif ablation_case == 2:
        # print("Ablation case 2: symmetrical widths.")
        fix_exponent  = False
        fix_width_symmetry_per_peak = fix_second_peak = True
    elif ablation_case == 3:
        # print("Ablation case 3: fixed exponent = 2.")
        fix_exponent = True
        fix_second_peak = fix_width_symmetry_per_peak = False
    elif ablation_case == 4:
        # print("Ablation case 4: fixed exponent = 2 and antipodal preferred angles.")
        fix_exponent = fix_second_peak = True
        fix_width_symmetry_per_peak = False
    elif ablation_case == 5:
        # print("Ablation case 5: fixed exponent = 2 and antipodal and individually symmetrical widths.")
        fix_exponent = fix_width_symmetry_per_peak = fix_second_peak = True

    def init_params_from(x, y):
        if ablation_case == None:
            p0 = hypothesis_engine.compute_initial_params(
                param_estimator=parameter_estimator,
                neuron_model=neuron_model,
                x=np.asarray(x),
                y=np.asarray(y),
            ).copy()
        else:
            p0 = hypothesis_engine.compute_initial_params(
                param_estimator=(lambda stimuli, spike_counts: parameter_estimator(stimuli, spike_counts, fix_exponent=fix_exponent, fix_second_peak=fix_second_peak, fix_width_symmetry_per_peak=fix_width_symmetry_per_peak)),
                neuron_model=neuron_model,
                x=np.asarray(x),
                y=np.asarray(y),
            ).copy()
        if manual_exponents is not None:
            p0 = p0.at[:, 5].set(manual_exponents)
            p0 = p0.at[:, 10].set(manual_exponents)
        
        if ablation_case == 1:
            # Ablation case 1 : set the second peak to equal the first peak + pi (mod 2pi). Do this by removing the 6th parameter (second orientation)
            p0 = jnp.delete(p0, 6, axis=1)
        elif ablation_case == 2:
            # Ablation case 2 : symmetrical widths.  
            width_ccw_1 = p0[:, 3]
            width_cw_1 = p0[:, 4]
            width_avg_1 = (width_ccw_1 + width_cw_1) / 2
            p0 = p0.at[:, 3].set(width_avg_1)
            # p0 = p0.at[:, 4].set(width_avg_1)

            width_ccw_2 = p0[:, 8]
            width_cw_2 = p0[:, 9]
            width_avg_2 = (width_ccw_2 + width_cw_2) / 2
            p0 = p0.at[:, 8].set(width_avg_2)
            # p0 = p0.at[:, 9].set(width_avg_2)
            
            # also have to delete the theta_pref_2 parameter (6th param)
            p0 = jnp.delete(p0, jnp.array([6, 4, 9]), axis=1)
        elif ablation_case == 3:
            # Ablation case 3 : fixed exponent = 2. Remove the 6th and 11th parameters (exponents)
            p0 = jnp.delete(p0, jnp.array([5, 10]), axis=1)
        elif ablation_case == 4:
            # Ablation case 4 : fixed exponent = 2 and antipodal preferred angles. Remove the 6th and 11th parameters (exponents) and set the second peak to equal the first peak + pi (mod 2pi). Do this by removing the 6th parameter (second orientation)
            p0 = jnp.delete(p0, jnp.array([5, 6, 10]), axis=1)
        elif ablation_case == 5:
            # Ablation case 5 : fixed exponent = 2 and antipodal and individually symmetrical widths. Remove the 6th and 11th parameters (exponents) and set the second peak to equal the first peak + pi (mod 2pi). Do this by removing the 6th parameter (second orientation)
            width_ccw_1 = p0[:, 3]
            width_cw_1 = p0[:, 4]
            width_avg_1 = (width_ccw_1 + width_cw_1) / 2
            p0 = p0.at[:, 3].set(width_avg_1)
            # p0 = p0.at[:, 4].set(width_avg_1)

            width_ccw_2 = p0[:, 8]
            width_cw_2 = p0[:, 9]
            width_avg_2 = (width_ccw_2 + width_cw_2) / 2
            p0 = p0.at[:, 8].set(width_avg_2)
            # p0 = p0.at[:, 9].set(width_avg_2)

            p0 = jnp.delete(p0, jnp.array([5, 6, 10, 4, 9]), axis=1)            
        return p0

    # ============================================================
    # CASE 1: No val sets → single-phase training
    # ============================================================
    if val1_fraction == 0 and val2_fraction == 0:
        print("\n=== Single-phase training on ALL data ===\n")
        if default_init_params is not None:
            print(f'Using default params for initialization.')
            # params = jnp.array([[0.0, 5.0, 20.0, 1.0, 1.0, 2.0,
            #                      3.14, 5.0, 20.0, 1.0, 1.0, 2.0]] * x_data.shape[0])
            params = default_init_params
        else:
            print(f'Using data-derived params for initialization.')
            params = init_params_from(x_data, y_data)
        n_cells, n_params = params.shape
        init_opt, train_step = make_train_fns(x_data, y_data)
        opt_state = init_opt(params)
        params_dynamic = np.zeros((num_steps, n_cells, n_params), dtype=np.float32)
        for step in range(1, num_steps + 1):
            params, opt_state, loss_val = train_step(params, opt_state)
            params_dynamic[step - 1] = params
            if step % print_every == 0:
                print(f"Step {step:4d} | Loss(all): {float(loss_val):.4f}")
        return {
            "params": params,
            "loss": np.array(loss_total(params, x_data, y_data)),
            "params_dynamic": params_dynamic,
            "data_splits": {
                "train_trials": training_trials,
                "val1_trials": val1_trials,
                "val2_trials": val2_trials,
            },
        }

    # ============================================================
    # Phase A: Train on train
    # ============================================================
    print("\n=== Phase A: Train on TRAIN only ===\n")
    params_A = init_params_from(x_train, y_train)
    n_cells, n_params = params_A.shape
    if debug :
        print(f"Number of cells: {n_cells}, number of params: {n_params}")
    init_opt_A, train_step_A = make_train_fns(x_train, y_train)
    opt_state_A = init_opt_A(params_A)

    params_dynamic_A = np.zeros((num_steps, n_cells, n_params), dtype=np.float32)
    val1_loss_dynamic = np.zeros((num_steps, n_cells), dtype=np.float32)

    val1_loss_cells = jax.jit(lambda p: loss_total(p, x_val1, y_val1))
    for step in range(1, num_steps + 1):
        params_A, opt_state_A, train_loss_val = train_step_A(params_A, opt_state_A)
        params_dynamic_A[step - 1] = params_A
        val1_loss_dynamic[step - 1] = np.array(val1_loss_cells(params_A))
        if step % print_every == 0:
            print(f"Step {step:4d} | Train mean: {float(train_loss_val):.4f} | "
                  f"Val1 mean: {float(np.nanmean(val1_loss_dynamic[step - 1])):.4f}")

    # ============================================================
    # Phase B: Patience selection (per-cell or global)
    # ============================================================
    print("\n=== Phase B: Patience selection ===\n")
    patience_values = [0]
    best_steps_val1 = np.argmin(val1_loss_dynamic, axis=0)
    patience_per_cell = np.zeros(n_cells, dtype=int)

    # Helper for per-cell loss
    val2_loss_single = jax.jit(lambda p, x, y: jnp.nanmean(loss_function(neuron_model(x, *p), y)))

    if val2_fraction > 0:
        if cellwise_patience:
            # ----- per-cell patience -----
            for i in range(n_cells):
                metrics = []
                for p in patience_values:
                    step_idx = int(np.clip(best_steps_val1[i] + p, 0, num_steps - 1))
                    params_tmp = params_dynamic_A[step_idx, i]
                    metric = float(val2_loss_single(params_tmp, x_val2[i], y_val2[i]))
                    metrics.append(metric)
                patience_per_cell[i] = patience_values[int(np.argmin(metrics))]
            print(f"Cellwise patience selected (range {min(patience_per_cell)}–{max(patience_per_cell)})")
        else:
            # ----- global patience -----
            mean_metrics = []
            for p in patience_values:
                step_idxs = np.clip(best_steps_val1 + p, 0, num_steps - 1)
                params_tmp = np.array([params_dynamic_A[step_idxs[i], i] for i in range(n_cells)])
                loss_val2 = np.array([val2_loss_single(params_tmp[i], x_val2[i], y_val2[i]) for i in range(n_cells)])
                mean_metrics.append(np.nanmean(loss_val2))
            best_p = patience_values[int(np.argmin(mean_metrics))]
            patience_per_cell[:] = best_p
            print(f"Global patience selected: {best_p}")
    else:
        # fallback: use val1 again if no val2
        for i in range(n_cells):
            metrics = [val1_loss_dynamic[int(np.clip(best_steps_val1[i] + p, 0, num_steps - 1)), i]
                       for p in patience_values]
            patience_per_cell[i] = patience_values[int(np.argmin(metrics))]
        print("Patience selected using val1 (no val2 set).")

    best_steps_final = np.clip(best_steps_val1 + patience_per_cell, 0, num_steps - 1)
    print(f'Mean best_steps_final: {np.mean(best_steps_final)}')

    # ============================================================
    # Phase C: Retrain on ALL data
    # ============================================================
    print("\n=== Phase C: Retrain on ALL data ===\n")
    if default_init_params is not None:
        print(f'Using default params for initialization.')
        # params_C0 = jnp.array([[0.0, 5.0, 20.0, 1.0, 1.0, 2.0,
        #                         3.14, 5.0, 20.0, 1.0, 1.0, 2.0]] * x_data.shape[0])
        # params_C0 = jnp.array([default_init_params] * x_data.shape[0])
        params_C0 = default_init_params
    else:
        print(f'Using data-derived params for initialization.')
        params_C0 = init_params_from(x_data, y_data)
    initial_params = params_C0.copy()
    init_opt_C, train_step_C = make_train_fns(x_data, y_data)
    opt_state_C = init_opt_C(params_C0)
    params_dynamic_C = np.zeros((num_steps, n_cells, n_params), dtype=np.float32)
    params_C = params_C0

    for step in range(1, num_steps + 1):
        params_C, opt_state_C, total_loss_val = train_step_C(params_C, opt_state_C)
        params_dynamic_C[step - 1] = params_C
        if step % print_every == 0:
            print(f"Step {step:4d} | Loss(all): {float(total_loss_val):.4f}")

    # Clip just in case
    best_steps_final = np.clip(best_steps_final, 0, num_steps - 1)
    final_params = params_dynamic_C[best_steps_final, np.arange(n_cells)]
    final_loss = np.array(loss_total(final_params, x_data, y_data))

    return {
        "params": final_params,
        "loss": final_loss,
        "params_dynamic": params_dynamic_C,
        "data_splits": {
            "train_trials": training_trials,
            "val1_trials": val1_trials,
            "val2_trials": val2_trials,
        },
        "best_steps": best_steps_final,
        "patience_per_cell": patience_per_cell,
        "cellwise_patience": cellwise_patience,
        "initial_params": initial_params if return_params_init else None,
    }

In [19]:
neuron_model_vmap = jax.vmap(lambda a, p: neuron_model(a, *p), in_axes=(0, 0))
neuron_model_gauss_vmap = jax.vmap(lambda a, p: neuron_model_2_jax(a, *p), in_axes=(0, 0))
neuron_model_antipodal_vmap = jax.vmap(lambda a, p: neuron_model_antipodal(a, *p), in_axes=(0, 0))
neuron_model_antipodal_symmetric_vmap = jax.vmap(lambda a, p: neuron_model_antipodal_symmetric(a, *p), in_axes=(0, 0))

In [31]:
import math 

def train_all_ablation_cases(
    dataset_name,
    response, 
    x_data,
    y_data,
    manual_exponents=None,
):
    ''' Train the model for all ablation cases to determine the final parameters and return the results in a dictionary. 

    Returns:
    --------
    eigenvalues_result : dict
        A dictionary with keys as the ablation case (Gaussian, Antipodal, Antipodal + Symmetric Widths, Full Model) and the eigenvalues as values 
    '''
    eigenvalues_result = {}

    # First calculate the data eigenspectrum
    n_repeats, n_cells, n_bins = response.shape
    window_size = n_bins // 8
    angles = x_data[0]
    angles_padded = np.concatenate([angles - 2*np.pi, angles, angles + 2*np.pi])
    angles_padded = angles_padded[len(angles)-window_size : 2*len(angles)+window_size]

    R_for_cv_pca = response.copy()
    n_blocks = len(R_for_cv_pca)
    R_padded = np.concatenate([R_for_cv_pca] * 3, axis=2)
    R_padded = R_padded[:, :, len(angles)-window_size : 2*len(angles)+window_size]
    R_smooth = np.zeros_like(R_for_cv_pca)

    n_cells = R_for_cv_pca[0].shape[0]

    best_fracs = np.zeros((n_cells, n_blocks))
    frac_values = np.array([0.01, 0.02, 0.05, 0.075, 0.1])

    for session in range(n_blocks):
        preds_padded, fracs = cv_pca.fit_lowess(
            R_full=R_padded[session],
            theta_full=angles_padded,
            frac_values=frac_values,
            n_folds=2
        )
        best_fracs[:, session] = fracs
        R_smooth[session] = preds_padded[:, window_size:-window_size]

    evals_est_three_rep_cvpca = np.zeros(len(angles))
    for i in range(n_blocks):
        j = (i + 1) % n_blocks
        k = (i + 2) % n_blocks
        l = (i + 3) % n_blocks

        evals_est_three_rep_cvpca += three_rep_cvpca(R_smooth[i], R_for_cv_pca[j], R_for_cv_pca[k]) / (n_blocks * math.comb(n_blocks - 1, 2))
        evals_est_three_rep_cvpca += three_rep_cvpca(R_smooth[i], R_for_cv_pca[k], R_for_cv_pca[l]) / (n_blocks * math.comb(n_blocks - 1, 2))
        evals_est_three_rep_cvpca += three_rep_cvpca(R_smooth[i], R_for_cv_pca[l], R_for_cv_pca[j]) / (n_blocks * math.comb(n_blocks - 1, 2))

    eigenvalues_result['data'] = evals_est_three_rep_cvpca

    # Fit the Gaussian model first 
    results = train_generalised(
        x_data=x_data,
        y_data=y_data,
        neuron_model=neuron_model_2_jax,
        parameter_estimator=parameter_estimator_2,
        loss_function=quadratic_loss,
        learning_rate=1e-3,
        val1_fraction=0.2,
        debug=True,
        num_steps=6_000,
        ablation_case=None,
    )
    pred = neuron_model_gauss_vmap(x_data, results['params'])
    ss = three_rep_cvpca(pred, pred, pred)
    eigenvalues_result['gaussian'] = ss
    
    for ablation_case in [None, 1, 2]:
        if ablation_case is None:
            nm = neuron_model
        elif ablation_case == 1:
            nm = neuron_model_antipodal
        elif ablation_case == 2:
            nm = neuron_model_antipodal_symmetric

        results = train_generalised(
            x_data=x_data,
            y_data=y_data,
            neuron_model=nm,
            parameter_estimator=parameter_estimator_optimised,
            loss_function=loss_functions.quadratic_loss,
            learning_rate=1e-3,
            val1_fraction=0.2,
            debug=True,
            num_steps=6_000,
            manual_exponents=manual_exponents,
            ablation_case=ablation_case,
        )

        if ablation_case is None:
            pred = neuron_model_vmap(x_data, results['params'])
            ss = three_rep_cvpca(pred, pred, pred)
            eigenvalues_result['ai_model'] = ss
        elif ablation_case == 1:
            pred = neuron_model_antipodal_vmap(x_data, results['params'])
            ss = three_rep_cvpca(pred, pred, pred)
            eigenvalues_result['antipodal'] = ss
        elif ablation_case == 2:
            pred = neuron_model_antipodal_symmetric_vmap(x_data, results['params'])
            ss = three_rep_cvpca(pred, pred, pred)
            eigenvalues_result['antipodal_symmetric'] = ss

    return eigenvalues_result


def compute_r_squared(A_test, R_test, params, nm):
    ''' Compute the r squared values for the AI model and Gaussian model on the test set. '''
    nm_vmap = jax.vmap(lambda a, p: nm(a, *p), in_axes=(0, 0))
    R_test_pred = nm_vmap(A_test, params)
    r_squared_values = r_squared(R_test, R_test_pred)
    return r_squared_values

def compute_r_squared_all_models(A_train, R_train, A_test, R_test, manual_exponents=None):
    ''' Compute the r squared values for all models on the test set. '''
    r_squared_results = {}

    neuron_model_param_estimator_tuple = {
        'gauss_r_squared' : (neuron_model_2_jax, parameter_estimator_2),
        'r_squared_antipodal' : (neuron_model_antipodal, parameter_estimator_optimised),
        'r_squared_antipodal_sym' : (neuron_model_antipodal_symmetric, parameter_estimator_optimised),
        'ai_r_squared' : (neuron_model, parameter_estimator_optimised),
    }

    manual_exponents = get_manual_exponents(dataset_name, R_train, A_train)
    # manual_exponents = None
    for model_name, (nm, pe) in neuron_model_param_estimator_tuple.items():
        ablation_case = 0 if model_name == 'ai_r_squared' else 1 if model_name == 'r_squared_antipodal' else 2 if model_name == 'r_squared_antipodal_sym' else None

        results = train_generalised(
            x_data=A_train,
            y_data=R_train,
            neuron_model=nm,
            parameter_estimator=pe,
            loss_function=loss_functions.quadratic_loss,
            learning_rate=1e-3,
            val1_fraction=0.2,
            debug=True,
            num_steps=6_000,
            ablation_case=ablation_case,
            manual_exponents=manual_exponents,
        )

        params = results['params']
        r_squared_values = compute_r_squared(A_test, R_test, params, nm)
        r_squared_results[model_name] = r_squared_values

    return r_squared_results


In [21]:
def parse_data(response, seed=42):
    rng = np.random.default_rng(seed=seed)
    n_repeats, n_cells, n_trials = response.shape
    n_train_repeats = 4
    n_test_repeats = n_repeats - n_train_repeats
    train_repeats = rng.choice(n_repeats, size=n_train_repeats, replace=False)
    test_repeats = np.setdiff1d(np.arange(n_repeats), train_repeats)

    # define train/test splits
    R_train = response[train_repeats].mean(axis=0)
    R_test = response[test_repeats].mean(axis=0)
    R_all = response.mean(axis=0)
    
    return(R_train, R_test, R_all)

In [32]:
def neuron_model_1_jax(theta, theta_pref=0.0, baseline=0.0, amplitude=1.0, tuning_width=1.0):
    theta_pref = jnp.clip(theta_pref, 0, 2 * jnp.pi)
    baseline = jnp.clip(baseline, 0, None)
    amplitude = jnp.clip(amplitude, 0, None)
    tuning_width = jnp.clip(tuning_width, 0.01, None)
    circ_dist_rad = lambda theta1, theta2: jnp.abs(jnp.arctan2(jnp.sin(theta1 - theta2), jnp.cos(theta1 - theta2)))
    dist = circ_dist_rad(theta, theta_pref)
    return baseline + amplitude * jnp.exp(-0.5 * (dist / tuning_width) ** 2)

def parameter_estimator_1(theta, spike_counts):
    """
    Estimates the parameters of the gaussian neuron model. We do this by creating a binned tuning curve and picking out salient features.
    Args:
        theta (np.ndarray): Angles in radians.
        spike_counts (np.ndarray): Spike counts corresponding to each angle.
    Returns:
        np.ndarray: Estimated parameters [theta_pref, baseline, amplitude, tuning_width].
    """
    n_bins = 20
    bin_idx = ((theta * n_bins) / (2 * np.pi)).astype(np.int32)
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)
    sums = np.bincount(bin_idx, weights=spike_counts, minlength=n_bins)
    counts = np.bincount(bin_idx, minlength=n_bins)
    tuning_curve = np.zeros(n_bins, dtype=np.float32)
    tuning_curve[counts > 0] = sums[counts > 0] / counts[counts > 0]
    pref_idx = np.argmax(tuning_curve)
    theta_pref = pref_idx * (2 * np.pi / n_bins)
    baseline = np.min(tuning_curve)
    amplitude = np.max(tuning_curve) - baseline
    half_max = baseline + amplitude / 2.0
    indices = (np.arange(-5, 6) + pref_idx) % n_bins
    above_half_max = tuning_curve[indices] >= half_max
    full_width_half_max = 2 * np.pi * np.sum(above_half_max) / n_bins
    tuning_width = full_width_half_max / (2.0 * np.sqrt(2 * np.log(2)))
    return np.array([theta_pref, baseline, amplitude, tuning_width])

def neuron_model_single(theta,
                        theta_pref=0.0,
                        baseline=0.0,
                        amplitude=1.0,
                        width_ccw=1.0,
                        width_cw=1.0,
                        exponent=2.0):
    """
    Single-peak circularly tuned neuron model with asymmetric widths and generalized Gaussian exponent.
    """
    min_width = 5e-2
    eps = 1e-12
    min_exponent, max_exponent = 0.1, 5.0

    # Parameter clipping
    width_ccw, width_cw = jnp.clip(width_ccw, min_width, None), jnp.clip(width_cw, min_width, None)
    exponent = jnp.clip(exponent, min_exponent, max_exponent)
    baseline = jnp.clip(baseline, 0.0, None)
    amplitude = jnp.clip(amplitude, 0.0, None)

    # Circular signed difference
    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))

    signed_diff = _signed_circ_diff_rad(theta, theta_pref) + eps
    width_effective = jnp.where(signed_diff < 0, width_ccw, width_cw)
    width_effective = jnp.maximum(width_effective, 1e-6)

    # Generalized Gaussian shape
    peak_component = amplitude * jnp.exp(-0.5 * (jnp.abs(signed_diff) / width_effective) ** exponent)
    return baseline + peak_component

def parameter_estimator_single(stimuli, spike_counts):
    """
    Estimate parameters for the single-peak neuron model.

    Parameters:
        stimuli (np.ndarray): Angles (radians, 0 to 2π)
        spike_counts (np.ndarray): Spike counts

    Returns:
        np.ndarray: [theta_pref, baseline, amplitude, width_ccw, width_cw, exponent]
    """
    n_bins = 256
    kernel_sigma = 2.5
    min_model_width = 1e-6
    default_width = 1.0
    min_exponent, max_exponent = 0.1, 5.0
    default_exponent = 2.0
    min_peak_amplitude = 0.5

    # --- Binning ---
    bin_idx = ((stimuli * n_bins) / (2 * np.pi)).astype(np.int32)
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    sums = np.bincount(bin_idx, weights=spike_counts, minlength=n_bins)
    counts = np.bincount(bin_idx, minlength=n_bins)

    # --- Smoothing (circular Gaussian) ---
    kernel_radius = int(3 * kernel_sigma)
    x = np.arange(-kernel_radius, kernel_radius + 1)
    kernel = np.exp(-0.5 * (x / kernel_sigma) ** 2)
    kernel /= np.sum(kernel) + 1e-8

    pad = len(kernel) // 2
    sums_padded = np.pad(sums, (pad, pad), mode='wrap')
    counts_padded = np.pad(counts, (pad, pad), mode='wrap')

    num_conv = np.convolve(sums_padded, kernel, mode='valid')
    den_conv = np.convolve(counts_padded, kernel, mode='valid')

    tuning_curve = np.zeros_like(num_conv)
    valid_mask = den_conv > 1e-8
    tuning_curve[valid_mask] = num_conv[valid_mask] / den_conv[valid_mask]

    angle_step = 2 * np.pi / n_bins

    # --- Baseline ---
    baseline = np.maximum(0.0, np.min(tuning_curve))

    # --- Peak detection ---
    peak_idx = np.argmax(tuning_curve)
    peak_val = tuning_curve[peak_idx]
    amplitude = peak_val - baseline

    if amplitude < min_peak_amplitude:
        return np.array([0.0, baseline, amplitude, default_width, default_width, default_exponent])

    # --- Estimate width & exponent ---
    target_half_val = baseline + amplitude / 2
    half_ccw_bins, half_cw_bins = 0, 0
    for k in range(1, n_bins // 2 + 1):
        if half_ccw_bins == 0 and tuning_curve[(peak_idx - k) % n_bins] <= target_half_val:
            half_ccw_bins = k
        if half_cw_bins == 0 and tuning_curve[(peak_idx + k) % n_bins] <= target_half_val:
            half_cw_bins = k
        if half_ccw_bins > 0 and half_cw_bins > 0:
            break

    SQRT_2_LOG_2 = np.sqrt(2 * np.log(2))
    width_ccw = (half_ccw_bins * angle_step) / SQRT_2_LOG_2 if half_ccw_bins > 0 else default_width
    width_cw = (half_cw_bins * angle_step) / SQRT_2_LOG_2 if half_cw_bins > 0 else default_width
    width_ccw = np.clip(width_ccw, min_model_width, np.pi)
    width_cw = np.clip(width_cw, min_model_width, np.pi)

    # Exponent from half vs quarter
    target_qtr_val = baseline + amplitude / 4
    qtr_ccw_bins, qtr_cw_bins = 0, 0
    for k in range(1, n_bins // 2 + 1):
        if qtr_ccw_bins == 0 and tuning_curve[(peak_idx - k) % n_bins] <= target_qtr_val:
            qtr_ccw_bins = k
        if qtr_cw_bins == 0 and tuning_curve[(peak_idx + k) % n_bins] <= target_qtr_val:
            qtr_cw_bins = k
        if qtr_ccw_bins > 0 and qtr_cw_bins > 0:
            break

    exponent_est = []
    if half_ccw_bins > 0 and qtr_ccw_bins > half_ccw_bins:
        exponent_est.append(np.log(2) / np.log(qtr_ccw_bins / half_ccw_bins))
    if half_cw_bins > 0 and qtr_cw_bins > half_cw_bins:
        exponent_est.append(np.log(2) / np.log(qtr_cw_bins / half_cw_bins))
    exponent = np.mean(exponent_est) if exponent_est else default_exponent
    exponent = np.clip(exponent, min_exponent, max_exponent)

    theta_pref = peak_idx * angle_step

    return np.array([theta_pref, baseline, amplitude, width_ccw, width_cw, exponent])

def train_simplified(
    x_data: jnp.ndarray,
    y_data: jnp.ndarray,
    neuron_model: Callable,
    parameter_estimator: Callable,
    loss_function: Callable,
    num_steps: int = 6_000,
    learning_rate: float = 1e-3,
    print_every: int = 200,
    exhaustive_exponents: Optional[Sequence[float]] = None,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    Single-phase training on ALL data.
    Optionally do an exhaustive multi-start search over initial exponents:
      - For each exponent 'e' in exhaustive_exponents, override params_init[:, 5] and [:, 10] = e
      - Train all starts in parallel (vectorized)
      - Pick, per cell, the start that gives the lowest loss

    Returns:
      - params: (n_cells, n_params) best params per cell
      - loss: (n_cells,) final loss per cell for the chosen start
      - all_losses: (n_starts, n_cells) losses per cell for every start
      - chosen_start_idx: (n_cells,) argmin over starts for each cell
      - chosen_exponent: (n_cells,) exponent chosen per cell
      - params_per_start: (n_starts, n_cells, n_params) final params for each start (useful for diagnostics)
    """
    rng = np.random.default_rng(seed)

    # ---------------------------
    # Loss helpers (per cell + batched)
    # ---------------------------
    # loss for one cell & one param row
    loss_single = lambda p, x, y: jnp.nanmean(loss_function(neuron_model(x, *p), y), axis=-1)
    # vectorize over cells: (n_cells, params) x (n_cells, ...) -> (n_cells,)
    loss_total = jax.vmap(loss_single, in_axes=(0, 0, 0), out_axes=0)
    # mean over cells
    def mean_loss(params, x, y):
        return jnp.nanmean(loss_total(params, x, y))
    mean_loss = jax.jit(mean_loss)

    # ---------------------------
    # Optimizer setup (vectorized over starts)
    # ---------------------------
    opt = optax.adam(learning_rate)

    def make_train_fns(x, y):
        # loss for a single start (averaged over cells)
        def obj_single_start(params_single):
            return mean_loss(params_single, x, y)

        # vectorize value_and_grad over starts dimension
        v_obj_and_grad = jax.vmap(jax.value_and_grad(obj_single_start), in_axes=0, out_axes=(0, 0))

        def init_state(p0_starts):
            # p0_starts: (n_starts, n_cells, n_params)
            return jax.vmap(opt.init)(p0_starts)

        @jax.jit
        def step(p_starts, opt_state):
            # losses per start, grads per start
            losses, grads = v_obj_and_grad(p_starts)
            updates, opt_state = jax.vmap(opt.update)(grads, opt_state, p_starts)
            p_starts = jax.vmap(optax.apply_updates)(p_starts, updates)
            return p_starts, opt_state, losses
        return init_state, step

    # ---------------------------
    # Initialize params (base init once)
    # ---------------------------
    def init_params_all(x, y):
        p0 = compute_initial_params(
            param_estimator=parameter_estimator,
            neuron_model=neuron_model,
            x=np.asarray(x),
            y=np.asarray(y),
        ).copy()
        return p0  # (n_cells, n_params)

    params_init = init_params_all(x_data, y_data)  # (n_cells, n_params)
    n_cells, n_params = params_init.shape

    # Build starts: either 1 start (no exhaustive) or one per exponent
    if exhaustive_exponents is None or len(exhaustive_exponents) == 0:
        starts = 1
        p0_starts = params_init[None, ...]  # (1, n_cells, n_params)
        exps_arr = jnp.array([jnp.nan])     # placeholder
    else:
        exps_arr = jnp.asarray(exhaustive_exponents, dtype=params_init.dtype)  # (n_starts,)
        starts = int(exps_arr.shape[0])
        # tile and set exponent columns 5 and 10
        p0_starts = jnp.repeat(params_init[None, ...], repeats=starts, axis=0)  # (n_starts, n_cells, n_params)
        p0_starts = p0_starts.at[:, :, 5].set(exps_arr[:, None])
        p0_starts = p0_starts.at[:, :, 10].set(exps_arr[:, None])

    # ---------------------------
    # Train all starts in parallel on ALL data
    # ---------------------------
    init_opt, train_step = make_train_fns(x_data, y_data)
    opt_state = init_opt(p0_starts)
    params_starts = p0_starts

    for step in range(1, num_steps + 1):
        params_starts, opt_state, losses_per_start = train_step(params_starts, opt_state)
        if (print_every is not None) and (print_every > 0) and (step % print_every == 0):
            # report mean loss across starts for a quick sanity check
            print(f"Step {step:5d} | mean(loss over starts) = {float(jnp.nanmean(losses_per_start)):.6f}")

    # ---------------------------
    # Evaluate per cell, per start & pick the best start for each cell
    # ---------------------------
    # losses per start per cell: (n_starts, n_cells)
    loss_per_start_per_cell = jax.vmap(lambda p: loss_total(p, x_data, y_data))(params_starts)

    # argmin over starts for each cell
    best_start_idx = jnp.nanargmin(loss_per_start_per_cell, axis=0)  # (n_cells,)

    # gather best params per cell from params_starts (n_starts, n_cells, n_params)
    gather_idx = jnp.arange(n_cells)
    best_params = params_starts[best_start_idx, gather_idx, :]  # (n_cells, n_params)

    # final per-cell loss for chosen start
    final_loss = loss_total(best_params, x_data, y_data)  # (n_cells,)

    # chosen exponent per cell (nan if no exhaustive search)
    chosen_exponent = jnp.where(
        jnp.isnan(exps_arr[0]),
        jnp.full((n_cells,), jnp.nan, dtype=best_params.dtype),
        exps_arr[best_start_idx]
    )

    return {
        "params": np.array(best_params),
        "loss": np.array(final_loss),
        "all_losses": np.array(loss_per_start_per_cell),    # (n_starts, n_cells)
        "chosen_start_idx": np.array(best_start_idx),       # (n_cells,)
        "chosen_exponent": np.array(chosen_exponent),       # (n_cells,)
        "params_per_start": np.array(params_starts),        # (n_starts, n_cells, n_params)
    }

In [33]:
def get_manual_exponents(dataset_name, R_train, A_train):
    # if dataset_name.startswith('ali_grating') or dataset_name == 'jacob_BZ016_2025-06-24':
    if dataset_name.startswith('ali_grating') :
        n_cells, n_bins = R_train.shape
        R_double = (R_train[:, :n_bins//2] + R_train[:, n_bins//2:]) / 2.0
        angles_double = A_train[0][::2]

        results_double = train_simplified(
            x_data=np.repeat(angles_double[None, :], n_cells, axis=0),
            y_data=R_double,
            neuron_model=neuron_model_single,
            parameter_estimator=parameter_estimator_single,
            loss_function=loss_functions.quadratic_loss,
            num_steps=10_000,
            learning_rate=1e-3,
            exhaustive_exponents=np.arange(0.5, 3.25, 0.25)
        )
        return results_double['params'][:, 5]
    else:
        print("Skipping double wrapped data fitting for other datasets.")
        return None

In [40]:
def r_squared(y_true, y_pred):
    ss_res = jnp.sum((y_true - y_pred) ** 2, axis=1)
    ss_tot = jnp.sum((y_true - jnp.mean(y_true, axis=1, keepdims=True)) ** 2, axis=1)
    return 1.0 - ss_res / jnp.maximum(ss_tot, 1e-8)

In [34]:
new_per_dataset_ss_dict = {}
new_per_dataset_dict = {}

In [41]:
mouse = 'BZ015'
date = '2025-07-03'
# mouse = 'BZ016'
# date = '2025-06-24'
exp_nums = [2, 3, 5] if mouse == 'BZ015' else [1]

dataset_name = f'jacob_{mouse}_{date}'

data_dirs = []
metadata_dirs = []

for n_exp in exp_nums:
    spks_path = f"/home/dabin/data/jacob_gratings_202507/parsed/{mouse}_{date}_{n_exp}"
    stims_path = f"/home/dabin/data/jacob_gratings_202507/parsed/{mouse}_{date}_{n_exp}"

    spks_file = f"{spks_path}/{mouse}_{date}_{n_exp}_dspikes.npy"
    stims_file = f"{stims_path}/{date}_{n_exp}_{mouse}_Block.mat"

    data_dirs.append(spks_file)
    metadata_dirs.append(stims_file)

response, angles = utils.load_data(data_dir = [data_dirs, metadata_dirs],
                                   data_type='jacob',
                                   conc_thresh=0.4,
                                   activity_thresh=0.0,
                                   signal_fraction_thresh=0.875 if mouse == 'BZ015' else 0.9,
                                   n_bins=256, min_repeats=6 if mouse == 'BZ015' else 5,)

R_train, R_test, R_all = parse_data(response)
n_cells = response.shape[1]
A_train = A_test = A_all = np.repeat(angles[None, :], n_cells, axis=0)

new_per_dataset_dict[dataset_name] = compute_r_squared_all_models(A_train, R_train, A_test, R_test)

new_per_dataset_ss_dict[dataset_name] = train_all_ablation_cases(
    dataset_name,
    response,
    x_data = A_all,
    y_data = R_all,
    manual_exponents=get_manual_exponents(dataset_name, R_all, A_all),
)

Selected 1907 / 15166 cells with activity > 0.0 and concentration > 0.4.
Selected 457 / 1907 cells with signal fraction > 0.875.
Skipping double wrapped data fitting for other datasets.
x_data shape : (457, 256), y_data shape : (457, 256)
Data split: 205 train, 51 val1, 0 val2

=== Phase A: Train on TRAIN only ===

Number of cells: 457, number of params: 5
Step  100 | Train mean: 0.1067 | Val1 mean: 0.1050
Step  200 | Train mean: 0.0957 | Val1 mean: 0.0947
Step  300 | Train mean: 0.0894 | Val1 mean: 0.0887
Step  400 | Train mean: 0.0848 | Val1 mean: 0.0843
Step  500 | Train mean: 0.0813 | Val1 mean: 0.0809
Step  600 | Train mean: 0.0786 | Val1 mean: 0.0782
Step  700 | Train mean: 0.0764 | Val1 mean: 0.0762
Step  800 | Train mean: 0.0747 | Val1 mean: 0.0747
Step  900 | Train mean: 0.0734 | Val1 mean: 0.0734
Step 1000 | Train mean: 0.0723 | Val1 mean: 0.0725
Step 1100 | Train mean: 0.0714 | Val1 mean: 0.0718
Step 1200 | Train mean: 0.0707 | Val1 mean: 0.0712
Step 1300 | Train mean: 0.070

NameError: name 'cv_pca' is not defined

In [ ]:
mouse = 'BZ016'
date = '2025-06-24'
exp_nums = [2, 3, 5] if mouse == 'BZ015' else [1]

dataset_name = f'jacob_{mouse}_{date}'

data_dirs = []
metadata_dirs = []

for n_exp in exp_nums:
    spks_path = f"/home/dabin/data/jacob_gratings_202507/parsed/{mouse}_{date}_{n_exp}"
    stims_path = f"/home/dabin/data/jacob_gratings_202507/parsed/{mouse}_{date}_{n_exp}"

    spks_file = f"{spks_path}/{mouse}_{date}_{n_exp}_dspikes.npy"
    stims_file = f"{stims_path}/{date}_{n_exp}_{mouse}_Block.mat"

    data_dirs.append(spks_file)
    metadata_dirs.append(stims_file)

response, angles = utils.load_data(data_dir = [data_dirs, metadata_dirs],
                                   data_type='jacob',
                                   conc_thresh=0.4,
                                   activity_thresh=0.0,
                                   signal_fraction_thresh=0.875 if mouse == 'BZ015' else 0.9,
                                   n_bins=256, min_repeats=6 if mouse == 'BZ015' else 5,)


R_train, R_test, R_all = parse_data(response)
n_cells = response.shape[1]
A_train = A_test = A_all = np.repeat(angles[None, :], n_cells, axis=0)

new_per_dataset_dict[dataset_name] = compute_r_squared_all_models(A_train, R_train, A_test, R_test)
new_per_dataset_ss_dict[dataset_name] = train_all_ablation_cases(
    dataset_name,
    response,
    x_data = A_all,
    y_data = R_all,
    manual_exponents=None
)

In [ ]:
dataset_names = ['gratings_drifting_GT3_2019_04_05_1', 'gratings_drifting_GT1_2019_04_12_1', 'gratings_drifting_GT2_2019_04_05_1']

for dataset_name in dataset_names:
    data_dir = f'/home/dabin/data/stringer_2020/{dataset_name}.npy'
    response, angles = utils.load_data(data_dir = data_dir,
                                    data_type='stringer',
                                    conc_thresh=0.4,
                                    activity_thresh=0.0,
                                    signal_fraction_thresh=0.8,
                                    n_bins=256, min_repeats=6)

    R_train, R_test, R_all = parse_data(response)
    n_cells = response.shape[1]
    A_train = A_test = A_all = np.repeat(angles[None, :], n_cells, axis=0)

    new_per_dataset_dict[dataset_name] = compute_r_squared_all_models(A_train, R_train, A_test, R_test)
    new_per_dataset_ss_dict[dataset_name] = train_all_ablation_cases(
        dataset_name,
        response,
        x_data = A_all,
        y_data = R_all,
        manual_exponents=get_manual_exponents(dataset_name, R_all, A_all),
    )


In [ ]:
dataset_name = 'ali_grating_M01'
stim_dir = '/home/dabin/data/ali_valentin_2025/stim_sequence.npy'
resp_dir = '/home/dabin/data/ali_valentin_2025/stim_resps.npy'

response, angles = utils.load_data(data_dir = [stim_dir, resp_dir],
                                   data_type='ali',
                                   conc_thresh=0.45,
                                   activity_thresh=0.0,
                                   signal_fraction_thresh=0.9,
                                   n_bins=90, min_repeats=6)

R_train, R_test, R_all = parse_data(response)
n_cells = response.shape[1]
A_train = A_test = A_all = np.repeat(angles[None, :], n_cells, axis=0)

new_per_dataset_dict[dataset_name] = compute_r_squared_all_models(A_train, R_train, A_test, R_test)
new_per_dataset_ss_dict[dataset_name] = train_all_ablation_cases(
    dataset_name,
    response,
    x_data = A_all,
    y_data = R_all,
    manual_exponents=get_manual_exponents(dataset_name, R_all, A_all),
)


In [ ]:
colour_map = ['tab:green', 'tab:olive', 'tab:orange', 'tab:red']

In [ ]:
import pandas as pd
# for each dataset, calculate the mean difference in explained variance between the gaussian vs the other 3 models (antipodal sym, antipodal, ai)
mean_diffs = {}
for dataset_name, dataset_results in new_per_dataset_dict.items():
    # if 'jacob' in dataset_name or 'GT3' in dataset_name:
    #     continue
    # Get the explained variance for each model
    gaussian_var = dataset_results['gauss_r_squared']
    antipodal_sym_var = dataset_results['r_squared_antipodal_sym']
    antipodal_var = dataset_results['r_squared_antipodal']
    ai_var = dataset_results['ai_r_squared']

    # Calculate the mean differences
    mean_diffs[dataset_name] = {
        'gaussian' : 0,
        'antipodal_sym': np.mean(antipodal_sym_var - gaussian_var),
        'antipodal': np.mean(antipodal_var - gaussian_var),
        'ai': np.mean(ai_var - gaussian_var)
    }    

# create a bar plot with the x axis as the different models. Each bar shows you the mean mean_diff across datasets, with the actual mean value of each dataset shown as a dot on the bar that is colour coded.
# models = regime_names
models = ['gaussian', 'antipodal_sym', 'antipodal', 'ai']
# datasets = list(mean_diffs.keys())
datasets = sorted(mean_diffs.keys())

# Gather per-dataset values into array
values = np.array([[mean_diffs[d][m] for m in models] for d in datasets])
mean_across_datasets = values.mean(axis=0)

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(models))

# Bars (mean across datasets)
bars = ax.bar(x, mean_across_datasets, color=[colour_map[m] for m in np.arange(len(models))], edgecolor='black', width=0.6, zorder=1, alpha=0.7)

# Dots + lines per dataset, sort the dataset in alphabetical order
datasets.sort()
colors = plt.cm.tab10(np.linspace(0, 1, len(datasets)))
for i, dataset in enumerate(datasets):
    ax.plot(x, values[i, :], color=colors[i], marker='o', linewidth=1.5, label=dataset_name_map[dataset], zorder=3)

# Style and labels
ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(['Gaussian', 'Gaussian + p', 'Gaussian + p \n + asym', 'Gaussian + p \n+ asym + non-antipodal\n (AI Model)'], fontsize=12)
ax.set_ylabel('Mean Δ Explained Variance across cells (vs Gaussian)', fontsize=14)
ax.set_title('Model Performance Improvement over Gaussian', fontsize=16)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(title='Dataset', loc='upper left', frameon=True, fontsize=11)
# ax.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, )
ax.grid(axis='y', linestyle='--', zorder=0)
plt.tight_layout()
plt.show()
